# K-Means Clustering Model Training

In this notebook, we'll train a K-Means clustering model to segment our customers into groups.

**What is K-Means?** K-Means is a simple method that groups similar customers together. We need to decide how many groups (clusters) to create. Too few groups and we lose detail. Too many and they become meaningless.

**Our approach:**

1. Test different numbers of clusters (2 to 10)
2. Use the "Elbow Method" to find the sweet spot
3. Check with "Silhouette Score" to validate our choice
4. Train the final model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import pickle

# Load RFM features
rfm_data = pd.read_csv('../data/processed/rfm_features.csv')

print(f"Loaded {len(rfm_data)} customers")

rfm_data.head()

ModuleNotFoundError: No module named 'src'

In [ ]:
# Prepare data for clustering

X = rfm_data[['Recency', 'Frequency', 'Monetary']].values

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Data prepared: {X_scaled.shape}")
print("Data standardized")

best_k = 3
model = train_kmeans(X, k=best_k)
silhouette = calculate_silhouette(X, k=best_k)
{'chosen_k': best_k, 'silhouette': silhouette}

In [ ]:
# Step 3: Find the best number of clusters using the Elbow Method

# Test different cluster numbers - Elbow Method

inertias = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    print(f"k={k}: inertia={kmeans.inertia_:.2f}")

# Plot
plt.figure(figsize=(10, 5))
plt.plot(list(k_range), inertias, marker='o', linewidth=2)
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.grid(True, alpha=0.3)
plt.show()

print("\n✓ Look for the 'elbow' (bend) in the curve above")

In [ ]:
# Step 4: Check with Silhouette Score (validates cluster quality)
from sklearn.metrics import silhouette_score

# Check silhouette scores

silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"k={k}: silhouette={score:.3f}")

# Plot
plt.figure(figsize=(10, 5))
plt.plot(list(k_range), silhouette_scores, marker='s', linewidth=2, color='green')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score')
plt.grid(True, alpha=0.3)
plt.show()

best_k = list(k_range)[np.argmax(silhouette_scores)]
print(f"\nBest k: {best_k}")

In [ ]:
# Step 5: Train the final K-Means model
# Based on the elbow method and silhouette score, choose optimal k

# Choose k based on what you see in the plots above
# Common choices: 3, 4, or 5 clusters
optimal_k = 4

print(f"Training K-Means with k={optimal_k}...")

# Train final model - use k=4 (good balance)
final_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = final_model.fit_predict(X_scaled)

# Add cluster labels to our data
rfm_data['Cluster'] = cluster_labels

print(f"✓ Model trained successfully!")
print(f"✓ Cluster assignments: {np.bincount(cluster_labels)}")

# Show cluster distribution
print("\n=== CLUSTER DISTRIBUTION ===")
for i in range(optimal_k):
    count = (cluster_labels == i).sum()
    percentage = (count / len(cluster_labels)) * 100
    print(f"Cluster {i}: {count} customers ({percentage:.1f}%)")

In [ ]:
# Step 6: Save the model and clustered data

import pickle

# Save model and data
with open('../models/kmeans_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

rfm_data.to_csv('../data/processed/rfm_with_clusters.csv', index=False)

print("✓ Model saved")
print("✓ Ready for visualization")